# Kiểm tra phát hiện mùi cà phê trong `data/hardware`

Mỗi UUID là một lượt đo. Nhãn cà phê lấy từ tên mẫu trong metadata có từ riêng `CF`, `cà phê`, `cafe`, `càfé`, `café` hoặc `coffee`. Tên `TaivsCF` cũng thuộc nhóm cà phê. Các tên khác và lượt chưa đặt tên được ghi là **Chưa xác nhận**.

Notebook chạy bộ phát hiện cà phê của web app trên các lượt đã lưu hoàn chỉnh. Bảng cuối hiển thị từng lượt, gồm cả lượt chưa có dự đoán và lý do. Tỷ lệ phát hiện tính trên nhóm có nhãn cà phê. Model chưa có danh sách UUID huấn luyện để xác minh các lượt này độc lập với tập huấn luyện.

Chọn kernel Python có `ipykernel` trong VS Code và bấm **Run All**. Chạy từ thư mục dự án hoặc một thư mục con của dự án.

In [1]:
import json
import re
import sys
from collections import Counter
from html import escape
from pathlib import Path

from IPython.display import Markdown, display

cwd = Path.cwd().resolve()
root = next(
    (path for path in (cwd, *cwd.parents)
     if (path / "robonose_s3").is_dir() and (path / "data" / "hardware").is_dir()),
    None,
)
if root is None:
    raise FileNotFoundError("Hãy mở notebook từ thư mục dự án có data/hardware.")
sys.path.insert(0, str(root))

from robonose_s3.coffee_detection import load_model
from robonose_s3.inference import InferenceCoordinator

hardware = root / "data" / "hardware"
coffee_name = re.compile(r"(?:^|[\W_])(?:cf|cà\s+phê|cafe|càfé|café|coffee)(?=$|[\W_])")
results = []
seen = set()

for path in sorted(hardware.rglob("trials/*/metadata.json")):
    metadata = json.loads(path.read_text(encoding="utf-8-sig"))
    trial_id = metadata["trial_id"]
    if trial_id in seen:
        raise ValueError(f"UUID trùng {trial_id}: {path.parent}")
    seen.add(trial_id)

    name = metadata.get("name") or ""
    normalized = name.strip().casefold()
    labeled_coffee = normalized == "taivscf" or bool(coffee_name.search(normalized))
    trial_status = metadata.get("status", "Không rõ")

    predicted_coffee = None
    if trial_status != "COMPLETE" or not metadata.get("export_finalized"):
        reason = f"Lượt chưa lưu hoàn chỉnh ({trial_status})."
    else:
        prediction = InferenceCoordinator._check_coffee(path.with_name("raw.csv"))
        reason = prediction.get("reason", "")
        if prediction["status"] == "PREDICTED":
            predicted_coffee = prediction["has_coffee"]

    if predicted_coffee is None:
        outcome = "Chưa dự đoán"
    elif not labeled_coffee:
        outcome = "Chưa chấm"
    else:
        outcome = "Đúng" if predicted_coffee else "Bỏ sót"

    results.append({
        "name": name or "(chưa đặt tên)",
        "trial_id": trial_id,
        "folder": path.parent.relative_to(hardware).as_posix(),
        "trial_status": trial_status,
        "labeled_coffee": labeled_coffee,
        "predicted_coffee": predicted_coffee,
        "outcome": outcome,
        "reason": reason,
    })

if not results:
    raise FileNotFoundError(f"Không tìm thấy metadata của lượt đo trong {hardware}.")

In [2]:
coffee_results = [result for result in results if result["labeled_coffee"]]
counts = Counter(result["outcome"] for result in coffee_results)
total_coffee = len(coffee_results)
correct = counts["Đúng"]
rate = f"{correct / total_coffee:.1%}" if total_coffee else "chưa có mẫu"
total_predicted = sum(result["predicted_coffee"] is not None for result in results)

display(Markdown(
    f"**{len(results)} lượt đo** · **{total_predicted}** có dự đoán · "
    f"**{len(results) - total_predicted}** chưa dự đoán.  \n"
    f"Nhóm có nhãn cà phê: **{correct}/{total_coffee} đúng ({rate})**, "
    f"**{counts['Bỏ sót']}** bỏ sót, **{counts['Chưa dự đoán']}** chưa dự đoán.  \n"
    f"**{len(results) - total_coffee}** lượt chưa xác nhận nhãn qua tên mẫu. "
    f"Model `{load_model()['version']}`."
))

def markdown_cell(value):
    return escape(" ".join(str(value).splitlines())).replace("|", r"\|")

prediction_labels = {True: "Có cà phê", False: "Không có cà phê", None: "—"}
table = [
    "| Tên mẫu | Thư mục lượt | Trạng thái lượt | Nhãn theo tên mẫu | Dự đoán | Đánh giá | Lý do |",
    "| --- | --- | --- | --- | --- | --- | --- |",
]
for result in results:
    fields = (
        result["name"],
        Path(result["folder"]).name,
        result["trial_status"],
        "Có cà phê" if result["labeled_coffee"] else "Chưa xác nhận",
        prediction_labels[result["predicted_coffee"]],
        result["outcome"],
        result["reason"] or "—",
    )
    table.append("| " + " | ".join(map(markdown_cell, fields)) + " |")

display(Markdown("\n".join(table)))

**32 lượt đo** · **29** có dự đoán · **3** chưa dự đoán.  
Nhóm có nhãn cà phê: **4/6 đúng (66.7%)**, **2** bỏ sót, **0** chưa dự đoán.  
**26** lượt chưa xác nhận nhãn qua tên mẫu. Model `logistic_regression_v1`.

| Tên mẫu | Thư mục lượt | Trạng thái lượt | Nhãn theo tên mẫu | Dự đoán | Đánh giá | Lý do |
| --- | --- | --- | --- | --- | --- | --- |
| kk | kk_060129_2026-11-23_6d5c9fcd3d59 | COMPLETE | Chưa xác nhận | Có cà phê | Chưa chấm | — |
| Phucluu | Phucluu_060730_2026-11-23_0e3e2d1a423c | COMPLETE | Chưa xác nhận | Không có cà phê | Chưa chấm | — |
| Phucluu | Phucluu_061711_2026-11-23_00e04a6f81c9 | COMPLETE | Chưa xác nhận | Không có cà phê | Chưa chấm | — |
| Phucluu | Phucluu_062545_2026-11-23_852e9296b978 | COMPLETE | Chưa xác nhận | Không có cà phê | Chưa chấm | — |
| Phucluu | Phucluu_063525_2026-11-23_ff7458e9b174 | COMPLETE | Chưa xác nhận | Không có cà phê | Chưa chấm | — |
| (chưa đặt tên) | Chua_dat_ten_064511_2026-11-23_4a4a0b430c17 | ABORTED | Chưa xác nhận | — | Chưa dự đoán | Lượt chưa lưu hoàn chỉnh (ABORTED). |
| (chưa đặt tên) | Chua_dat_ten_065826_2026-11-23_81be29c4075a | COMPLETE | Chưa xác nhận | Có cà phê | Chưa chấm | — |
| cc | cc_072234_2026-11-23_31984efe84d2 | ABORTED | Chưa xác nhận | — | Chưa dự đoán | Lượt chưa lưu hoàn chỉnh (ABORTED). |
| (chưa đặt tên) | Chua_dat_ten_072803_2026-11-23_2a64bb73f321 | ABORTED | Chưa xác nhận | — | Chưa dự đoán | Lượt chưa lưu hoàn chỉnh (ABORTED). |
| Phucluu | Phucluu_074818_2026-11-23_afa070e7f04d | COMPLETE | Chưa xác nhận | Không có cà phê | Chưa chấm | — |
| Phucluu | Phucluu_080016_2026-11-23_b6d5a3217539 | COMPLETE | Chưa xác nhận | Không có cà phê | Chưa chấm | — |
| Duong_E205 | Duong_E205_105650_2026-11-20_e2fa1697d43e | COMPLETE | Chưa xác nhận | Không có cà phê | Chưa chấm | — |
| Duong_E205 | Duong_E205_110453_2026-11-20_a1e3e39866c0 | COMPLETE | Chưa xác nhận | Không có cà phê | Chưa chấm | — |
| Tai_E205 | Tai_E205_102319_2026-11-20_d7eaa1dcf1dc | COMPLETE | Chưa xác nhận | Không có cà phê | Chưa chấm | — |
| Tai_E205 | Tai_E205_104612_2026-11-20_533aa57e7657 | COMPLETE | Chưa xác nhận | Không có cà phê | Chưa chấm | — |
| Duong_KTX | Duong_KTX_171319_2026-11-10_a3d739c7960d | COMPLETE | Chưa xác nhận | Không có cà phê | Chưa chấm | — |
| Duong_KTX | Duong_KTX_173005_2026-11-10_e20df7180e45 | COMPLETE | Chưa xác nhận | Không có cà phê | Chưa chấm | — |
| khongkhi_KTX | khongkhi_KTX_164347_2026-11-10_62f68654454a | COMPLETE | Chưa xác nhận | Không có cà phê | Chưa chấm | — |
| khongkhi_KTX | khongkhi_KTX_165023_2026-11-10_656c7794a42e | COMPLETE | Chưa xác nhận | Không có cà phê | Chưa chấm | — |
| khongkhi_KTX | khongkhi_KTX_170136_2026-11-10_b75789c9eb5e | COMPLETE | Chưa xác nhận | Không có cà phê | Chưa chấm | — |
| test | test_162243_2026-11-10_0c9b3d63a68f | COMPLETE | Chưa xác nhận | Không có cà phê | Chưa chấm | — |
| test | test_163111_2026-11-10_c5f595e065b8 | COMPLETE | Chưa xác nhận | Có cà phê | Chưa chấm | — |
| CF_KTX | CF_KTX_205310_2026-11-10_6660d4a270df | COMPLETE | Có cà phê | Có cà phê | Đúng | — |
| CF_KTX | CF_KTX_210622_2026-11-10_8ab84c345c11 | COMPLETE | Có cà phê | Có cà phê | Đúng | — |
| CF_KTX | CF_KTX_212139_2026-11-10_9d7d07d44c94 | COMPLETE | Có cà phê | Không có cà phê | Bỏ sót | — |
| CF_KTX | CF_KTX_213437_2026-11-10_b2be80959fce | COMPLETE | Có cà phê | Không có cà phê | Bỏ sót | — |
| CF_KTX | CF_KTX_215301_2026-11-10_ed319df2db69 | COMPLETE | Có cà phê | Có cà phê | Đúng | — |
| Duong_CF_KTX | Duong_CF_KTX_221154_2026-11-10_ac2cf6e1479a | COMPLETE | Có cà phê | Có cà phê | Đúng | — |
| Duong_KTX | Duong_KTX_202410_2026-11-10_6209dffdc68b | COMPLETE | Chưa xác nhận | Có cà phê | Chưa chấm | — |
| Duong_KTX | Duong_KTX_203914_2026-11-10_4958f2dd190a | COMPLETE | Chưa xác nhận | Không có cà phê | Chưa chấm | — |
| khongkhi_KTX | khongkhi_KTX_195832_2026-11-10_298a9dd93c01 | COMPLETE | Chưa xác nhận | Không có cà phê | Chưa chấm | — |
| khongkhi_KTX | khongkhi_KTX_200856_2026-11-10_1070222ec9c1 | COMPLETE | Chưa xác nhận | Không có cà phê | Chưa chấm | — |